# Web and Text Analytics 2026-27 · Week 7 · Learning words

This notebook follows the slides of Week 7.

- **Part 1 · Meaning from use** repeats on the real reviews the numbers of Part I of the slides: the word-by-document and the word-by-word designs, the Euclidean distance and the cosine similarity.
- **Part 2 · Case study** has the five steps of Part IV. Each step has the same number as the tag *Notebook · Step N* on its slide.

Part 1 is optional: after the **Setup**, you can go straight to Part 2.

The data are the *Amazon Unlocked Mobile* reviews of Weeks 5 and 6, with the same split into 23,052 training and 7,684 test reviews.

**Runtime.** The free CPU runtime of Colab is enough. Step 1 trains word2vec in about 15 seconds. Step 4 uses a free Gemini API key; if you do not have one, the notebook gives the same step with an open model. Step 5 uses the open model, which is downloaded once (about 470 MB).

**Your numbers.** word2vec is trained with a fixed seed and one CPU core, so every run gives the same vectors, and your numbers should match the slides. With a different version of gensim they may differ in the second decimal; the words and the conclusions stay the same.

## Setup

In [ ]:
!pip install -q gensim google-genai

In [ ]:
import re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

### Load the data

Choose how to load the file `Amazon_Unlocked_Mobile.csv`:

- `"kaggle"`: the notebook downloads it from Kaggle with `kagglehub` (about 33 MB).
- `"upload"`: you upload the file from your computer (about 130 MB).

In [ ]:
SOURCE = "kaggle"   # or "upload"

if SOURCE == "kaggle":
    import kagglehub, glob
    path = kagglehub.dataset_download("nehasontakke/amazon-unlocked-mobilecsv")
    CSV = glob.glob(path + "/**/*.csv", recursive=True)[0]
else:
    from google.colab import files
    uploaded = files.upload()
    CSV = next(iter(uploaded))

df = pd.read_csv(CSV)
df.shape

### The same split as in Weeks 5 and 6

A 10% sample of the reviews, without missing values and without the 3-star reviews. A review is positive when it has 4 or 5 stars.

In [ ]:
sample = df.sample(frac=0.1, random_state=10).dropna()
sample = sample[sample['Rating'] != 3]
sample['Positively Rated'] = np.where(sample['Rating'] > 3, 1, 0)

X_train, X_test, y_train, y_test = train_test_split(sample['Reviews'], sample['Positively Rated'], random_state=0)
len(X_train), len(X_test)

# Part 1 · Meaning from use

### 1.1 The word-by-document design

A word is described by the documents in which it appears: its vector is its **column** of the document-term matrix of Week 5, one number per training review.

In [ ]:
vect = CountVectorizer().fit(X_train)
D = vect.transform(X_train)                 # reviews × words
vocab = vect.vocabulary_                    # word -> column
D.shape

In [ ]:
def column(word):
    return D[:, vocab[word]].toarray().ravel()

pairs = [('dies', 'drains'), ('excellent', 'excelent'), ('good', 'great'), ('good', 'bad')]
by_document = [cosine_similarity([column(a)], [column(b)])[0, 0] for a, b in pairs]
pd.DataFrame({'pair': [f'{a} – {b}' for a, b in pairs], 'word-by-document': np.round(by_document, 3)})

Two synonyms such as *dies* and *drains* rarely appear in the same review, so their columns share almost nothing and their cosine is close to 0.

### 1.2 The word-by-word design

A word is described by the words within **k = 2** positions before and after it. We count these co-occurrences in the training reviews; each word gets one row of the co-occurrence matrix.

In [ ]:
import scipy.sparse as sp

tok = re.compile(r'(?u)\b\w\w+\b')          # the same tokenizer as CountVectorizer
k = 2
rows, cols = [], []
for text in X_train:
    ids = [vocab[w] for w in tok.findall(text.lower()) if w in vocab]
    for i, w in enumerate(ids):
        for j in range(max(0, i - k), min(len(ids), i + k + 1)):
            if j != i:
                rows.append(w); cols.append(ids[j])

W = sp.coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(vocab), len(vocab))).tocsr()
W.shape

In [ ]:
def row(word):
    return W[vocab[word]].toarray().ravel()

show = ['battery', 'fast', 'quickly', 'phone', 'price', 'love']
pd.DataFrame([[int(row(w)[vocab[c]]) for c in show] for w in ['dies', 'drains', 'great', 'excellent']],
             index=['dies', 'drains', 'great', 'excellent'], columns=show)

This is the table of the slide *The word-by-word design*: six of the 19,533 columns.

### 1.3 How close are two vectors?

The made-up example of the slides: counts of the words *battery* and *camera* in three reviews.

In [ ]:
A = np.array([2, 6])    # short camera review
B = np.array([8, 16])   # long camera review
Cc = np.array([12, 8])  # long battery review

euclid = lambda u, v: np.sqrt(((u - v) ** 2).sum())
cosine = lambda u, v: u @ v / (np.linalg.norm(u) * np.linalg.norm(v))

for name, u, v in [('A – B', A, B), ('B – C', B, Cc)]:
    print(f'{name}: Euclidean {euclid(u, v):.2f}, cosine {cosine(u, v):.2f}, angle {np.degrees(np.arccos(cosine(u, v))):.0f}°')

The Euclidean distance puts the long camera review closer to the battery review; the cosine compares only the direction and puts the two camera reviews together.

### 1.4 Measuring the two designs

In [ ]:
by_word = [cosine_similarity([row(a)], [row(b)])[0, 0] for a, b in pairs]
pd.DataFrame({'pair': [f'{a} – {b}' for a, b in pairs],
              'word-by-document': np.round(by_document, 2),
              'word-by-word (k = 2)': np.round(by_word, 2)})

Synonyms and misspellings become close only when a word is described by its neighbours. But *good* and *bad* are just as close: opposites share their contexts.

**Question.** Each row of `W` has 19,533 numbers. How many of them are zero for the word *drains*? Why is that a problem?

In [ ]:
(row('drains') == 0).mean()

# Part 2 · Case study

## Step 1 · Train word2vec on unlabelled reviews

The corpus is **the 23,052 training reviews**, the same as in Weeks 5 and 6. Their labels are not used, and the test reviews stay unseen until the evaluation.

| Parameter | What it sets |
|---|---|
| `vector_size=100` | d: every word gets a vector of 100 numbers |
| `window=5` | up to 5 context words on each side of the centre word |
| `min_count=5` | words seen fewer than 5 times get no vector |
| `sg=1` | the skip-gram task; `sg=0`, the default, trains CBOW |
| `negative=5` | each example compares the real word with 5 random words, not all the vocabulary |
| `seed=42`, `workers=1` | the same vectors in every run; `epochs=5` makes 5 passes over the corpus |

In [ ]:
from gensim.models import Word2Vec

tok = re.compile(r'(?u)\b\w\w+\b')          # the same tokenizer as CountVectorizer (defined here too, so Part 2 runs without Part 1)

# the 23,052 training reviews; their labels are not used
sents = [tok.findall(t.lower()) for t in X_train]

w2v = Word2Vec(sents, vector_size=100, window=5, min_count=5,
               sg=1, negative=5, epochs=5, seed=42, workers=1)
len(sents), len(w2v.wv)

Each word is now a row of the embedding matrix E, which gensim keeps in `w2v.wv`:

In [ ]:
print(w2v.wv.vectors.shape)       # E: one row per word, d = 100 columns
w2v.wv['battery'][:10]            # the first 10 of the 100 numbers of battery

## Step 2 · Nearest neighbours

In [ ]:
w2v.wv.most_similar('dies', topn=5)

In [ ]:
for a, b in [('good', 'great'), ('good', 'bad'), ('love', 'hate')]:
    print(f'{a} – {b}: {w2v.wv.similarity(a, b):.2f}')

**In class.** Pick two words of your own. Find one list of neighbours that makes sense and one that does not, and explain the second one with the distributional hypothesis.

In [ ]:
w2v.wv.most_similar('refund', topn=5)     # change the word

*Optional.* Word arithmetic: which word is to *apple* what *galaxy* is to *samsung*? And to *better* what *bad* is to *good*? The three input words are left out of the answer.

In [ ]:
print(w2v.wv.most_similar(positive=['galaxy', 'apple'], negative=['samsung'], topn=3))
print(w2v.wv.most_similar(positive=['bad', 'better'], negative=['good'], topn=3))

### Step 2b · A pretrained model: GloVe

GloVe (Stanford, 2014) is the second method of the slide *Three ways to learn word embeddings*. We do not train it: we download vectors that Stanford learned from **Wikipedia and news** (6 billion words, 400,000 words in the vocabulary, 50 numbers per word, about 66 MB). Then we compare the neighbours of the same words in the two models.

In [ ]:
import gensim.downloader as api
glove = api.load('glove-wiki-gigaword-50')
print(len(glove), 'words with', glove.vector_size, 'numbers each')

In [ ]:
for word in ['dies', 'excellent', 'refund', 'screen']:
    print(word)
    print('   GloVe, Wikipedia and news :', [w for w, s in glove.most_similar(word, topn=5)])
    print('   our word2vec, reviews     :', [w for w, s in w2v.wv.most_similar(word, topn=5)])
print('good – bad   GloVe:', round(float(glove.similarity('good', 'bad')), 2),
      '  our word2vec:', round(float(w2v.wv.similarity('good', 'bad')), 2))

**Questions.**
1. Why are the neighbours of *dies* so different in the two models? Which meaning of the word does each corpus see?
2. Is *good* close to *bad* in GloVe too? What does this tell you about opposites?
3. Which model would you use for our reviews, and which for news articles? Recall the slide *Creating word embeddings: the corpus*.

A word that the model never saw, or saw fewer than 5 times, has no vector:

In [ ]:
try:
    w2v.wv['excellenttt']
except KeyError as e:
    print('KeyError:', e)

## Step 3 · A map of our space

The vectors have 100 numbers, so we cannot draw them. **PCA** finds the two directions in which the vectors of the chosen words vary most and places each word along them.

In [ ]:
groups = {'Battery drain': ['dies', 'drains', 'drained', 'discharges', 'overheats'],
          'Praise': ['great', 'excellent', 'excelent', 'awesome', 'amazing', 'perfect'],
          'Returns and money': ['refund', 'return', 'refunded', 'exchange'],
          'The screen': ['screen', 'display', 'lcd', 'screens']}
words = [w for ws in groups.values() for w in ws]

pca = PCA(n_components=2)
xy = pca.fit_transform(w2v.wv[words])
print('Share of the spread kept by the two axes:', pca.explained_variance_ratio_.sum().round(2))

plt.figure(figsize=(10, 6.5))
i = 0
for name, ws in groups.items():
    pts = xy[i:i + len(ws)]; i += len(ws)
    plt.scatter(pts[:, 0], pts[:, 1], s=60, label=name)
    for w, (x, y) in zip(ws, pts):
        plt.annotate(w, (x, y), xytext=(5, 4), textcoords='offset points', fontsize=12)
plt.xlabel('First principal component'); plt.ylabel('Second principal component')
plt.legend(); plt.show()

The two axes keep only part of the spread of the 100 numbers, so two words that look close on the picture may not be close in 100 dimensions. The cosine in all 100 dimensions stays the measure.

**Try it.** Add the word `'bad'` to the group *Praise* and run the cell again. Where does it land, and why?

**Practise it.** Step 1 of the interactive exercise opens our model in the Embedding Projector, with all its 5,667 words.

## Step 4 · A text embedding model through an API

So far each **word** has a vector. To compare whole reviews we need one vector per **review**.

**The baseline.** The simplest review vector is the **average** of the vectors of its words (mean pooling); words without a vector are skipped.

In [ ]:
def review_vector(text):
    ws = [w for w in tok.findall(text.lower()) if w in w2v.wv]
    return w2v.wv[ws].mean(axis=0) if ws else np.zeros(w2v.vector_size)

for a, b in [('Battery dies fast.', 'The battery drains quickly.'),
             ('Battery dies fast.', 'Terrible power life.'),
             ('Not good, bad.', 'Good, not bad.')]:
    print(f'{a} – {b}: {cosine_similarity([review_vector(a)], [review_vector(b)])[0, 0]:.2f}')

The average brings together reviews that share no word (*dies* and *drains*), but it ignores the order of the words: “Not good, bad.” and “Good, not bad.” have the same words, so they get the same vector.

**A text embedding model** turns a whole text into one vector, and it reads every word in its context. We use Google's `gemini-embedding-001` through the Gemini API.

**Get a free API key (once).**
1. Open [Google AI Studio](https://aistudio.google.com/apikey), sign in with a Google account and click **Create API key**.
2. In Colab, click the **key icon** (Secrets) on the left, add a secret with the name `GEMINI_API_KEY`, paste the key as its value and allow this notebook to use it.

Never paste the key into a cell: anyone who sees the notebook could use it. The free tier allows about 1,000 embedded texts per day, and the texts you send go to Google. Our reviews are public Kaggle data.

*No key?* Skip the next two cells and run the cell **Without a key** below.

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

In [ ]:
texts = ['Battery dies fast.', 'Terrible power life.', 'Not good, bad.', 'Good, not bad.']
res = client.models.embed_content(model='gemini-embedding-001', contents=texts)
E = np.array([e.values for e in res.embeddings])      # 4 texts × 3,072 numbers
print(E.shape)

S_api = cosine_similarity(E)
S_w2v = cosine_similarity(np.vstack([review_vector(t) for t in texts]))
pd.DataFrame({'pair': [f'{texts[0]} – {texts[1]}', f'{texts[2]} – {texts[3]}'],
              'average of word2vec': [S_w2v[0, 1], S_w2v[2, 3]],
              'gemini-embedding-001': [S_api[0, 1], S_api[2, 3]]}).round(2)

**Without a key.** The same comparison with an open text embedding model, `intfloat/multilingual-e5-small`, which runs in Colab. This model expects the prefix `query: ` before each text. We need it in Step 5 as well.

In [ ]:
from sentence_transformers import SentenceTransformer

st = SentenceTransformer('intfloat/multilingual-e5-small')
texts = ['Battery dies fast.', 'Terrible power life.', 'Not good, bad.', 'Good, not bad.']
E = st.encode(['query: ' + t for t in texts], normalize_embeddings=True)
print(E.shape)

S_open = cosine_similarity(E)
S_w2v = cosine_similarity(np.vstack([review_vector(t) for t in texts]))
pd.DataFrame({'pair': [f'{texts[0]} – {texts[1]}', f'{texts[2]} – {texts[3]}'],
              'average of word2vec': [S_w2v[0, 1], S_w2v[2, 3]],
              'multilingual-e5-small': [S_open[0, 1], S_open[2, 3]]}).round(2)

**Questions.** Each model has its own scale of cosines, so compare the two rows *within* a column, not the numbers across the columns.
1. The average of word vectors gives 1.00 for “Not good, bad.” and “Good, not bad.”. Does the text embedding model give less?
2. In the column of the text embedding model, which pair is closer: two complaints written with different words, or two reviews with the same words and opposite meanings? What does this tell you about how well the model reads a negation?

## Step 5 · Semantic search

We search the **1,925 short test reviews** (3 to 12 words, duplicates removed) for the query *arrived broken* and show the top 3 of each representation: TF-IDF of Week 5, the average of word2vec and the open text embedding model. The free tier of the Gemini API allows about 1,000 texts per day, fewer than the 1,925 reviews, so here we use the open model.

In [ ]:
docs = list(pd.unique(pd.Series([t for t in X_test if 3 <= len(t.split()) <= 12])))
query = 'arrived broken'
print(len(docs), 'short reviews')

def top3(scores):
    return [docs[i] for i in np.argsort(-scores)[:3]]

# TF-IDF, fitted on the training reviews as in Week 5
tfidf = TfidfVectorizer().fit(X_train)
s_tfidf = cosine_similarity(tfidf.transform([query]), tfidf.transform(docs)).ravel()

# average of word2vec
D_w2v = np.vstack([review_vector(t) for t in docs])
s_w2v = cosine_similarity([review_vector(query)], D_w2v).ravel()

pd.DataFrame({'TF-IDF': top3(s_tfidf), 'average of word2vec': top3(s_w2v)})

The text embedding model. It is told which text is the query (`query: `) and which are the reviews (`passage: `). The cell takes about a minute.

In [ ]:
if 'st' not in globals():                       # if the cell Without a key of Step 4 was not run
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer('intfloat/multilingual-e5-small')

D_open = st.encode(['passage: ' + t for t in docs], normalize_embeddings=True, batch_size=64)
q_open = st.encode(['query: ' + query], normalize_embeddings=True)
s_open = cosine_similarity(q_open, D_open).ravel()

pd.DataFrame({'TF-IDF': top3(s_tfidf), 'average of word2vec': top3(s_w2v), 'multilingual-e5-small': top3(s_open)})

**Questions.**
1. Which representation returns only reviews that contain the word *broken*?
2. Which one returns reviews that are *praise* rather than a complaint? Why does the average of word vectors make this mistake?
3. Which one finds a complaint that does not contain the words of the query?
4. Try your own query, for example `'battery drains fast'`. Change `query` and run the cells of this step again.

## Where they stand

Compare your results with the table of the slide *Where they stand*: which representation sees synonyms, which sees the word order, and which tells a complaint from praise?

Start with the simplest representation, measure, and move to a heavier one only when the results justify it.